#### Saving Artifacts

This notebook saves the artifacts required for the app.

In [3]:
# Imports
import os, numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from joblib import dump

RESUMES_CSV = "data/processed/resumes_clean.csv"          
JDS_CSV     = "data/processed/jd_clean.csv"       
PAIRS_CSV   = "data/raw/gold_pairs_adzuna.csv"           

ART_DIR = "artifacts"
os.makedirs(ART_DIR, exist_ok=True)

# Original schema columns
RES_COL = "clean_text_redacted"
JD_COL  = "jd_text"
ID_COL  = "row_id"

In [5]:
# Load data & assemble training pairs
import numpy as np
import pandas as pd

resumes = pd.read_csv(RESUMES_CSV)
jds = pd.read_csv(JDS_CSV)
pairs = pd.read_csv(PAIRS_CSV)

# unify dtypes for joining
if ID_COL in resumes:
    resumes[ID_COL] = resumes[ID_COL].astype(str)
if ID_COL in pairs:
    pairs[ID_COL] = pairs[ID_COL].astype(str)
if "jd_id" in pairs and "jd_id" in jds:
    pairs["jd_id"] = pairs["jd_id"].astype(str)
    jds["jd_id"] = jds["jd_id"].astype(str)

# Pick the resume text column from resumes
RES_TEXT_COL = None
for c in ["clean_text_redacted", "clean_text", "raw_text", "resume_text"]:
    if c in resumes.columns:
        RES_TEXT_COL = c
        break

# Ensure jd_text exists in pairs: merge by jd_id if possible, else job_title
if "jd_text" not in pairs.columns:
    if "jd_id" in pairs and "jd_id" in jds.columns:
        jds_sub = jds[["jd_id", JD_COL]].dropna().drop_duplicates("jd_id")
        pairs = pairs.merge(jds_sub, on="jd_id", how="left")
    elif "job_title" in pairs.columns and "job_title" in jds.columns:
        # job_title may be non-unique in JDs, pick first occurrence per title
        jds_sub = jds[["job_title", JD_COL]].dropna().drop_duplicates("job_title")
        pairs = pairs.merge(jds_sub, on="job_title", how="left")
    else:
        raise AssertionError(
            "Cannot add jd_text: neither (jd_id in both) nor (job_title in both) is available to merge."
        )

# Merge resume text from resumes (fallback to pairs.resume_snippet if needed)
if RES_TEXT_COL is not None:
    pairs = pairs.merge(resumes[[ID_COL, RES_TEXT_COL]], on=ID_COL, how="left")
    pairs = pairs.rename(columns={RES_TEXT_COL: "resume_text"})
elif "resume_snippet" in pairs.columns:
    pairs = pairs.rename(columns={"resume_snippet": "resume_text"})
else:
    raise AssertionError(
        "No resume text found. Expected resumes.clean_text_redacted (or clean_text/raw_text) "
        "or pairs.resume_snippet as a fallback."
    )

# Clean up and keep only rows with both texts
pairs["resume_text"] = pairs["resume_text"].astype(str)
pairs["jd_text"] = pairs["jd_text"].astype(str)
pairs = pairs.replace({"None": np.nan, "nan": np.nan})
pairs = pairs.dropna(subset=["resume_text", "jd_text"]).copy()

# Ensure label; if missing or single-class, auto-generate negatives by mismatching resumes
if "label" not in pairs.columns or pairs["label"].nunique() == 1:
    rng = np.random.default_rng(42)
    pos = pairs[[ID_COL, "resume_text", "jd_text"]].drop_duplicates().copy()
    pos["label"] = 1
    shuf = pos.sample(frac=1.0, random_state=42).reset_index(drop=True)
    neg = pos.copy()
    neg["resume_text"] = shuf["resume_text"]  # mismatched resume → negative
    neg["label"] = 0
    pairs = pd.concat([pos, neg], ignore_index=True)

# Build the SAME join format the app uses
def join_pair(jd, rs):
    return f"[JD] {str(jd)}\n[RESUME] {str(rs)}"

pairs["pair_text"] = pairs.apply(lambda r: join_pair(r["jd_text"], r["resume_text"]), axis=1)
pairs = pairs.dropna(subset=["pair_text", "label"]).reset_index(drop=True)

print({
    "pairs_rows": len(pairs),
    "pos_rate": float(pairs["label"].mean()),
    "cols": list(pairs.columns)[:10]
})
pairs.head()


{'pairs_rows': 5575, 'pos_rate': 0.28035874439461883, 'cols': ['jd_id', 'job_title', 'row_id', 'score', 'resume_snippet', 'label', 'jd_text', 'resume_text', 'pair_text']}


,jd_id,job_title,row_id,score,resume_snippet,label,jd_text,resume_text,pair_text
0,0,Care Assistant,5297,4.757286,ENUMERATOR Summary Recent graduate with BA in ...,1,Company Description Location: Sutton Coldfield...,ENUMERATOR Summary Recent graduate with BA in ...,[JD] Company Description Location: Sutton Cold...
1,0,Care Assistant,8280,3.167249,Teacher Assistant Teacher Assistant Chicago IL...,0,Company Description Location: Sutton Coldfield...,Teacher Assistant Teacher Assistant Chicago IL...,[JD] Company Description Location: Sutton Cold...
2,0,Care Assistant,27839,0.000000,Technical Support Engineer Technical Support E...,0,Company Description Location: Sutton Coldfield...,Technical Support Engineer Technical Support E...,[JD] Company Description Location: Sutton Cold...
3,0,Care Assistant,23704,0.000000,Lab Support Lab Support IT Professional Cary N...,0,Company Description Location: Sutton Coldfield...,Lab Support Lab Support IT Professional Cary N...,[JD] Company Description Location: Sutton Cold...
4,0,Care Assistant,12361,0.000000,IT Cyber Risk Management Associate span lITspa...,0,Company Description Location: Sutton Coldfield...,IT Cyber Risk Management Associate span lITspa...,[JD] Company Description Location: Sutton Cold...


In [7]:
# TF-IDF tuned to avoid common English words in explanations/suggestions
vec = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),          
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
    max_features=200_000,        # keep memory in check
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9+\-_.#/&]{1,}\b"  # keeps tech tokens like c++, s3, etc.
)

X_train, X_val, y_train, y_val = train_test_split(
    pairs["pair_text"], pairs["label"], test_size=0.2, random_state=42, stratify=pairs["label"]
)

Xtr = vec.fit_transform(X_train)
Xva = vec.transform(X_val)
ytr = y_train.values
yva = y_val.values

In [9]:
# Linear model for interpretability (weights -> top positive/negative terms)
clf = LogisticRegression(
    penalty="l2",
    C=1.0,
    class_weight="balanced",
    solver="liblinear",
    max_iter=2000,
)
clf.fit(Xtr, ytr)

# Isotonic calibration on the *held-out* validation split
cal = CalibratedClassifierCV(clf, method="isotonic", cv="prefit")
cal.fit(Xva, yva)

CalibratedClassifierCV(cv='prefit',
                       estimator=LogisticRegression(class_weight='balanced',
                                                    max_iter=2000,
                                                    solver='liblinear'),
                       method='isotonic')

In [11]:
print("Train done. Shapes:", Xtr.shape, Xva.shape)

Train done. Shapes: (4460, 165893) (1115, 165893)


In [13]:
# Export artifacts (names your app expects)
dump(vec, os.path.join(ART_DIR, "vectorizer.joblib"))
dump(clf, os.path.join(ART_DIR, "logreg.joblib"))
dump(cal, os.path.join(ART_DIR, "calib_isotonic.joblib"))

print("Saved to:", ART_DIR)

Saved to: artifacts


In [15]:
# Smoke test: load back & score one example
from joblib import load
vec2 = load(os.path.join(ART_DIR, "vectorizer.joblib"))
lr2  = load(os.path.join(ART_DIR, "logreg.joblib"))
iso2 = load(os.path.join(ART_DIR, "calib_isotonic.joblib"))

jd_demo = "Data Analyst with Python, SQL, and Power BI; ETL pipelines; stakeholder dashboards."
rs_demo = "Built SQL data marts, automated ETL in Python, created Power BI dashboards with DAX."

pair_demo = f"[JD] {jd_demo}\n[RESUME] {rs_demo}"
Xd = vec2.transform([pair_demo])
raw = float(lr2.decision_function(Xd)[0])
p   = float(iso2.predict_proba(Xd)[:, 1][0])   # pass Xd, not raw
print({"raw": round(raw, 3), "calibrated_prob": round(p, 3)})

{'raw': -0.304, 'calibrated_prob': 0.247}
